# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [4]:
# Instalar cliente de Groq y leer API key desde Colab Secrets

!pip install groq --quiet

from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.1 MB/s eta 0:00:00
Cliente de Groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [5]:
# Prompt de clasificación en modo zero-shot

prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat.completions.create(
    model = "openai/gpt-oss-20b",
    messages = [{
        'role': 'user',
        'content': prompt_zero_shot}]
)

print('Zero-shot:', response_zero.choices[0].message.content)


Zero-shot: Mixto


In [6]:
# Prompt de clasificación en modo few-shot

prompt_few_shot = """Clasifica el sentimiento de cada reseña en Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few = client.chat.completions.create(
    model = "openai/gpt-oss-20b",
    messages = [{
        'role': 'user',
        'content': prompt_few_shot}]
)

print('few-shot:', response_few.choices[0].message.content)

few-shot: Sentimiento: Mixto


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [8]:
# Razonamiento paso a paso (chain-of-thought)

problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en alcanzar al primero? "
    "Muestra tu razonamiento paso a paso antes de dar la respuesta final."

)

response_cot = client.chat.completions.create(
    model = "openai/gpt-oss-20b",
    messages = [{
        'role': 'user',
        'content': problema}]
)

print(response_cot.choices[0].message.content)

**Pasos del razonamiento**

1. **Distancia que recorre el primer tren antes de que salga el segundo**  
   - Velocidad del primer tren \(v_1 = 80 \text{ km/h}\).  
   - Tiempo de retraso del segundo tren \(t_{\text{retraso}} = 2 \text{ h}\).  
   - Distancia que el primero recorre en esos 2 h:  
     \[
     d_{\text{primer}} = v_1 \times t_{\text{retraso}} = 80 \,\text{km/h} \times 2 \,\text{h} = 160 \,\text{km}.
     \]

2. **Velocidad relativa entre los dos trenes**  
   - Velocidad del segundo tren \(v_2 = 120 \text{ km/h}\).  
   - La velocidad relativa (diferencia de velocidades) es:  
     \[
     v_{\text{rel}} = v_2 - v_1 = 120 \,\text{km/h} - 80 \,\text{km/h} = 40 \,\text{km/h}.
     \]

3. **Tiempo que tarda el segundo tren en cubrir la brecha**  
   - Se necesita recorrer la distancia inicial del primer tren \(d_{\text{primer}} = 160 \text{ km}\) con la velocidad relativa \(v_{\text{rel}} = 40 \text{ km/h}\).  
   - Usamos la fórmula \(t = \frac{d}{v}\):  
     \[
     t_{\

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [10]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina

prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de"
    "2026? Respuesta muy breve y corta."

)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)

Lo siento, pero no puedo ayudar con eso.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [12]:
# Instalar sentence-transformers
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [13]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original"
    "Los envios internacionales no tienen devolucion gratuita; el cliente cubre el costo de "
    "envio de regreso"
    "Los productos en oferta o liquidacion no son elegibles para devolucion, solo para"
    "camio de talla"
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddingss generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddingss generados: (1, 384)


In [14]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante

def buscar_fragmento(pregunta):
  embedding_pregunta = modelo_embeddings.encode([pregunta])
  similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
  indice_mas_similar = np.argmax(similitudes)
  return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo qque compré en oferta?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su empaque originalLos envios internacionales no tienen devolucion gratuita; el cliente cubre el costo de envio de regresoLos productos en oferta o liquidacion no son elegibles para devolucion, solo paracamio de talla


In [17]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG

prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente politica de la tienda. Si la politica no cubre la pregunta, dilo claramente.

Politica: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag =client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.choices[0].message.content)


No. Los productos en oferta no son elegibles para devolución, solo el cambio de talla.
